## Data Cleaning


## 1. Import Libraries and Load Dataset

In [5]:
import pandas as pd
import numpy as np

df = pd.read_csv("C:\\TRAINING_CODES\\Online_Food_Delivery_Analysis\\Data\\ONLINE_FOOD_DELIVERY_ANALYSIS.csv")

df_clean = df.copy()

## 2. Initial Data Inspection


In [6]:
missing_percentage = (df_clean.isnull().sum() / len(df_clean)) * 100
missing_report = pd.DataFrame({
    "Missing Count": df_clean.isnull().sum(),
    "Missing Percentage": missing_percentage
})

missing_report = missing_report[missing_report["Missing Count"] > 0]
missing_report.sort_values(by="Missing Percentage", ascending=False)

,Missing Count,Missing Percentage
Cancellation_Reason,90969,90.969
Final_Amount,55697,55.697
Customer_Age,50093,50.093
Distance_km,33470,33.470
Delivery_Time_Min,33359,33.359
Order_Value,33327,33.327
Peak_Hour,32962,32.962
Customer_Gender,24856,24.856
Payment_Mode,19911,19.911
Cuisine_Type,16885,16.885


## 3. Data Cleaning
### 3.1 Missing Value Analysis

In [7]:
numeric_missing = [
    "Customer_Age",
    "Delivery_Time_Min",
    "Distance_km",
    "Order_Value",
    "Discount_Applied",
    "Final_Amount",
    "Delivery_Rating",
    "Restaurant_Rating",
    "Profit_Margin"
]

df_clean[numeric_missing].describe().T

,count,mean,std,min,25%,50%,75%,max
Customer_Age,49907.0,38.976516,12.372157,18.0,28.00,39.00,50.00,60.0
Delivery_Time_Min,66641.0,127.475923,90.805839,20.0,45.00,120.00,210.00,300.0
Distance_km,66530.0,16.449242,12.256742,1.0,5.47,9.97,27.43,40.0
Order_Value,66673.0,2081.830126,1553.628891,150.0,673.00,1197.00,3494.00,5000.0
Discount_Applied,83285.0,93.936243,108.209904,0.0,20.00,50.00,100.00,300.0
Final_Amount,44303.0,1961.101190,1557.354417,-150.0,559.00,1156.00,3375.00,4980.0
Delivery_Rating,83477.0,2.991531,1.414108,1.0,2.00,3.00,4.00,5.0
Restaurant_Rating,100000.0,4.249680,0.722554,3.0,3.60,4.20,4.90,5.5
Profit_Margin,100000.0,0.150362,0.201888,-0.2,-0.02,0.15,0.32,0.5


### 3.2 Numeric Missing Value Strategy

In [8]:
df_clean['Customer_Age'] = df_clean['Customer_Age'].fillna(df_clean['Customer_Age'].mean())
df_clean["Delivery_Time_Min"] = df_clean["Delivery_Time_Min"].fillna(df_clean["Delivery_Time_Min"].median())
df_clean["Distance_km"] = df_clean["Distance_km"].fillna(df_clean["Distance_km"].median())
df_clean["Order_Value"] = df_clean["Order_Value"].fillna(df_clean["Order_Value"].median())
df_clean["Discount_Applied"] = df_clean["Discount_Applied"].fillna(df_clean["Discount_Applied"].median())

### 3.3 Categorical Missing Value Strategy

In [9]:
df_clean["Customer_Gender"] = df_clean["Customer_Gender"].fillna("Unknown")
df_clean["City"] = df_clean["City"].fillna("Unknown")
df_clean["Area"] = df_clean["Area"].fillna("Unknown")
df_clean["Cuisine_Type"] = df_clean["Cuisine_Type"].fillna("Unknown")

df_clean["Payment_Mode"] = df_clean["Payment_Mode"].fillna(df_clean["Payment_Mode"].mode()[0])

### 3.4 Business Rule Validation
#### 3.4.1 Discount Validation

In [10]:
invalid_discount = df_clean["Discount_Applied"] > df_clean["Order_Value"]
print(invalid_discount.sum())

df_clean.loc[invalid_discount, "Discount_Applied"] = df_clean.loc[invalid_discount, "Order_Value"]

773


#### 3.4.2 Cancellation Reason Validation

In [11]:
df_clean.loc[(df_clean['Order_Status'] == 'Delivered') & (df_clean['Cancellation_Reason'].isnull()),'Cancellation_Reason'] = 'Not Cancelled'
df_clean.loc[(df_clean['Order_Status'] == 'Cancelled') & (df_clean['Cancellation_Reason'].isnull()),'Cancellation_Reason'] = 'Unknown'

#### 3.4.3 Restaurant Rating Validation

In [12]:
df_clean.loc[df_clean['Restaurant_Rating'] > 5, 'Restaurant_Rating'] = 5

#### 3.4.4 Profit Margin Validation

In [13]:
df_clean.loc[df_clean['Profit_Margin']<0, 'Profit_Margin'] = 0

### 3.5 Derived Column Correction

In [14]:
df_clean.loc[:, 'Final_Amount'] = df_clean['Order_Value'] - df_clean['Discount_Applied']

### 3.6 Date and Time Processing

In [15]:
df_clean["Order_Date"] = pd.to_datetime(df_clean["Order_Date"], errors="coerce")
df_clean["Order_Time"] = pd.to_datetime(df_clean["Order_Time"], errors="coerce")

df_clean = df_clean.dropna(subset=["Order_Date", "Order_Time"]) # Order_Time can't be filled so dropping rows with missing values in these columns

### 3.7 Delivery Rating Validation

In [16]:
df_clean.loc[(df_clean['Order_Status'] == 'Delivered') & (df_clean['Delivery_Rating'].isnull()), 'Delivery_Partner_ID'].count()
df_clean.loc[(df_clean['Order_Status'] == 'Cancelled') & (df_clean['Delivery_Rating'].isnull()), 'Delivery_Partner_ID'].count()

np.int64(2397)

## 4. Missing Value Audit
After preprocessing, only two columns contain missing values:
* **Delivery_Rating (16.49%)** — Missing values were intentionally preserved because a missing rating indicates that the customer did not submit a delivery review.
* **Peak_Hour (32.93%)** — Missing values were left unchanged because the source `Order_Time` column was corrupted (all values were `0:00`), making reliable derivation impossible.

In [17]:
cols = df_clean.columns[df_clean.isnull().any()].tolist()
df_missing_report = pd.DataFrame({
    "Column_Name": cols,
    "Missing Count": df_clean[cols].isnull().sum().values,
    "Missing Percentage": (df_clean[cols].isnull().sum().values / len(df_clean)) * 100
})
df_missing_report

,Column_Name,Missing Count,Missing Percentage
0,Delivery_Rating,15999,16.491944
1,Peak_Hour,31944,32.928225


In [18]:
rating = df_clean.loc[df_clean["Restaurant_Rating"] > 5, "Restaurant_Rating"].count()
profit = df_clean.loc[df_clean["Profit_Margin"] < 0, "Profit_Margin"].count()
discount = df_clean.loc[df_clean["Discount_Applied"] > df_clean["Order_Value"], "Discount_Applied"].count()
final_amount = df_clean.loc[df_clean["Final_Amount"] != df_clean["Order_Value"] - df_clean["Discount_Applied"], "Final_Amount"].count()

df_validation_report = pd.DataFrame({
    "Column_Name": ["Restaurant_Rating", "Profit_Margin", "Discount_Applied", "Final_Amount"],
    "Invalid_Values": [rating , profit, discount, final_amount],
    "Validation_Status": ["❌" if x > 0 else "✅" for x in [rating, profit, discount, final_amount]]
})
df_validation_report

,Column_Name,Invalid_Values,Validation_Status
0,Restaurant_Rating,0,✅
1,Profit_Margin,0,✅
2,Discount_Applied,0,✅
3,Final_Amount,0,✅


## 5. Final Audit

In [19]:
df_clean.info()

<class 'pandas.DataFrame'>
Index: 97011 entries, 0 to 99998
Data columns (total 25 columns):
 #   Column               Non-Null Count  Dtype         
---  ------               --------------  -----         
 0   Order_ID             97011 non-null  str           
 1   Customer_ID          97011 non-null  str           
 2   Customer_Age         97011 non-null  float64       
 3   Customer_Gender      97011 non-null  str           
 4   City                 97011 non-null  str           
 5   Area                 97011 non-null  str           
 6   Restaurant_ID        97011 non-null  str           
 7   Restaurant_Name      97011 non-null  str           
 8   Cuisine_Type         97011 non-null  str           
 9   Order_Date           97011 non-null  datetime64[us]
 10  Order_Time           97011 non-null  datetime64[us]
 11  Delivery_Time_Min    97011 non-null  float64       
 12  Distance_km          97011 non-null  float64       
 13  Order_Value          97011 non-null  float64   

## 6. Saving as a New File

In [20]:
df_clean.to_csv("CLEANED_ORDERS.csv", index=False)